# Regime-Shift Asset Allocation — Full Pipeline Walkthrough

This notebook runs the entire pipeline (`data -> features -> regime -> optimization -> backtest -> results`)
end to end and renders the charts inline. It's a thin wrapper around `scripts/run_pipeline.py` / `src/` —
all the real logic lives in the `src` package so it's tested and reusable outside a notebook.

Run `pip install -r requirements.txt` first if you haven't.

In [ ]:
import sys
from pathlib import Path
import yaml
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / 'notebooks').exists() else Path.cwd()))

from src import data as data_mod
from src import features as feat_mod
from src import backtest as bt_mod
from src import metrics as metrics_mod
from src import optimization as opt_mod
from src import plotting as plot_mod

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg

## 1. Data

In [ ]:
prices = data_mod.load_price_panel(
    tickers=cfg["data"]["tickers"],
    vix_ticker=cfg["data"].get("vix_ticker"),
    start=cfg["data"]["start"],
    end=cfg["data"].get("end"),
    cache_dir="../" + cfg["data"].get("cache_dir", "data_cache"),
)
asset_cols = list(cfg["data"]["tickers"].keys())
asset_returns = data_mod.prices_to_simple_returns(prices, asset_cols)
prices.tail()

In [ ]:
prices[asset_cols].plot(figsize=(13, 4.5), subplots=True, title="Asset prices");

## 2. Features

In [ ]:
feat_df = feat_mod.build_full_feature_matrix(
    asset_prices=prices,
    market_col="equity",
    vix=prices["vix"] if "vix" in prices.columns else None,
    momentum_windows=cfg["features"]["momentum_windows"],
    volatility_windows=cfg["features"]["volatility_windows"],
    vix_zscore_window=cfg["features"]["vix_zscore_window"],
)
combined = feat_df.join(asset_returns, rsuffix="_ret").dropna()
feat_df = combined.loc[:, feat_df.columns]
asset_returns = combined.loc[:, asset_cols]

stress_periods = [("2020-02-15", "2020-04-15"), ("2022-01-01", "2022-10-31")]
feat_mod.sanity_check_spikes(feat_df, "vol_21d", stress_periods)

## 3 & 4. Regime detection + regime-conditional optimization (walk-forward)

In [ ]:
wf_cfg = bt_mod.WalkForwardConfig(**cfg["walk_forward"])
opt_cfg = opt_mod.OptimizationConfig(
    max_weight=cfg["optimization"]["max_weight"],
    min_weight=cfg["optimization"]["min_weight"],
    risk_free_rate_annual=cfg["optimization"]["risk_free_rate"],
    bear_target_annual_return=cfg["optimization"]["bear_target_annual_return"],
)
regime_cfg = {
    "n_states": cfg["regime"]["n_states"],
    "covariance_type": cfg["regime"]["covariance_type"],
    "n_iter": cfg["regime"]["n_iter"],
    "random_state": cfg["regime"]["random_state"],
    "vol_feature_for_labeling": "vol_21d",
}

result = bt_mod.run_walk_forward_backtest(
    feat_df=feat_df,
    asset_returns=asset_returns,
    feature_columns=cfg["regime"]["feature_columns"],
    wf_cfg=wf_cfg,
    regime_cfg=regime_cfg,
    opt_cfg=opt_cfg,
    objectives=cfg["optimization"]["objectives"],
    tx_cost_bps=cfg["costs"]["transaction_cost_bps"],
)
result.regimes.value_counts()

In [ ]:
plot_mod.plot_price_with_regimes(
    prices.loc[result.regimes.index, "equity"], result.regimes,
    "Detected Regimes Overlaid on Equity Price (out-of-sample, walk-forward)"
);

In [ ]:
last_fold = max(result.fold_transition_matrices.keys())
plot_mod.plot_transition_matrix(
    result.fold_transition_matrices[last_fold],
    result.fold_state_labels[last_fold],
    f"Transition Matrix (final walk-forward fold #{last_fold})"
);

## 5. Backtest vs. benchmarks

In [ ]:
bench_6040 = bt_mod.static_benchmark_returns(asset_returns.loc[result.returns.index], cfg["benchmarks"]["static_6040"])
bench_eq = bt_mod.static_benchmark_returns(asset_returns.loc[result.returns.index], cfg["benchmarks"]["equal_weight"])

result_no_cost = bt_mod.run_walk_forward_backtest(
    feat_df=feat_df, asset_returns=asset_returns,
    feature_columns=cfg["regime"]["feature_columns"], wf_cfg=wf_cfg,
    regime_cfg=regime_cfg, opt_cfg=opt_cfg,
    objectives=cfg["optimization"]["objectives"], tx_cost_bps=0.0,
)

rebalances_per_year = 252 / max(1, cfg["walk_forward"]["min_holding_days"])
summary = pd.DataFrame({
    "Dynamic (net of costs)": metrics_mod.summarize(result.returns, result.weights_history,
                                                      cfg["optimization"]["risk_free_rate"],
                                                      rebalances_per_year=rebalances_per_year),
    "Dynamic (no costs)": metrics_mod.summarize(result_no_cost.returns, result_no_cost.weights_history,
                                                 cfg["optimization"]["risk_free_rate"],
                                                 rebalances_per_year=rebalances_per_year),
    "Static 60/40": metrics_mod.summarize(bench_6040, risk_free_annual=cfg["optimization"]["risk_free_rate"]),
    "Equal-Weight": metrics_mod.summarize(bench_eq, risk_free_annual=cfg["optimization"]["risk_free_rate"]),
}).T
summary

In [ ]:
plot_mod.plot_equity_curves({
    "Dynamic (net costs)": result.returns,
    "Dynamic (no costs)": result_no_cost.returns,
    "Static 60/40": bench_6040,
    "Equal-Weight": bench_eq,
}, "Out-of-Sample Equity Curves");

In [ ]:
plot_mod.plot_drawdown(result.returns, "Dynamic Strategy Drawdown");